# Part 16: Inference Acceleration: KV Cache and Serving Constraints

> **Previous context**: Generation chooses tokens one by one. Now we ask why that loop is slow and how systems speed it up.
> **Goal for this part**: Understand KV Cache, attention cost, batching, quantization, FlashAttention, and PagedAttention intuition.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Why generation is slow

Autoregressive models generate one token at a time. Recomputing all previous keys and values every step wastes work.

## 1. KV Cache

The cache stores past keys and values so each new token only computes the new pieces it needs.

## 2. Memory pressure

KV Cache saves compute but consumes memory. Serving systems must manage batch size, context length, and throughput.

## 3. System ideas

FlashAttention improves attention IO efficiency; PagedAttention manages KV memory more flexibly for many requests.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- KV Cache avoids recomputing past keys and values.
- Inference speed is limited by both compute and memory.
- Serving systems optimize batching, memory layout, and attention kernels.

Next, continue through the code cells for the Inference part and inspect the printed observations.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

torch.manual_seed(42)

In [2]:
# Teaching note: follow this line to see the main step.
def naive_generation_cost(n_tokens):
    'Read the values printed above and connect them to the concept in this cell.'
    return sum(range(1, n_tokens + 1))

for n in [10, 50, 100, 500]:
    cost = naive_generation_cost(n)
    print(f"Read the values printed above and connect them to the concept in this cell.{n:3d}Read the values printed above and connect them to the concept in this cell.{cost:6d}Read the values printed above and connect them to the concept in this cell.{cost - n:5d}Read the values printed above and connect them to the concept in this cell.")

print(f"Read the values printed above and connect them to the concept in this cell.{naive_generation_cost(100) - 100}Read the values printed above and connect them to the concept in this cell.")
print(f"Reason")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Reason

In [3]:
class AttentionWithKVCache(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        
        # Teaching note: follow this line to see the main step.
        self.k_cache = None
        self.v_cache = None
    
    def reset_cache(self):
        'Read the values printed above and connect them to the concept in this cell.'
        self.k_cache = None
        self.v_cache = None
    
    def forward(self, x, use_cache=True):
        'Read the values printed above and connect them to the concept in this cell.'
        batch_size, seq_len, _ = x.shape
        
        # Teaching note: follow this line to see the main step.
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        if use_cache:
            if self.k_cache is not None:
                # Teaching note: follow this line to see the main step.
                K = torch.cat([self.k_cache, K], dim=2)
                V = torch.cat([self.v_cache, V], dim=2)
            # Teaching note: follow this line to see the main step.
            self.k_cache = K
            self.v_cache = V
        
        # Teaching note: follow this line to see the main step.
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        weights = F.softmax(scores, dim=-1)
        attn_output = weights @ V
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_O(attn_output)

print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.

In [4]:
# Teaching note: follow this line to see the main step.
attn = AttentionWithKVCache(d_model=64, num_heads=4)

# Teaching note: follow this line to see the main step.
N = 20

# Teaching note: follow this line to see the main step.
start = time.time()
sequence = torch.randn(1, 1, 64)
for _ in range(N):
    _ = attn(sequence, use_cache=False)
    sequence = torch.cat([sequence, torch.randn(1, 1, 64)], dim=1)
no_cache_time = time.time() - start

# Teaching note: follow this line to see the main step.
attn.reset_cache()
start = time.time()
sequence = torch.randn(1, 1, 64)
for _ in range(N):
    _ = attn(sequence, use_cache=True)
    sequence = torch.randn(1, 1, 64)  # Teaching note: follow this line to see the main step.
cache_time = time.time() - start

print(f"Read the values printed above and connect them to the concept in this cell.{no_cache_time*1000:.1f} ms")
print(f"Read the values printed above and connect them to the concept in this cell.{cache_time*1000:.1f} ms")
print(f"Read the values printed above and connect them to the concept in this cell.{no_cache_time/cache_time:.1f}x")
print(f"Read the values printed above and connect them to the concept in this cell.")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

.
.
- .- Loss- Loss
.
.

In [ ]:
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.

num_layers = 32      # Teaching note: follow this line to see the main step.
num_heads = 32       # Teaching note: follow this line to see the main step.
head_dim = 128       # Teaching note: follow this line to see the main step.
# d_model = num_heads * head_dim = 4096

# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.

def kv_cache_bytes(seq_len, bytes_per_value):
    'Read the values printed above and connect them to the concept in this cell.'
    # Teaching note: follow this line to see the main step.
    return 2 * num_layers * num_heads * head_dim * seq_len * bytes_per_value

def format_size(bytes_size):
    'Read the values printed above and connect them to the concept in this cell.'
    if bytes_size >= 1024**3:
        return f"{bytes_size / 1024**3:.1f} GB"
    else:
        return f"{bytes_size / 1024**2:.0f} MB"

# Teaching note: follow this line to see the main step.
precisions = {
    "FP16": 2,
    "INT8": 1,
    "INT4": 0.5,
}

# Teaching note: follow this line to see the main step.
context_lengths = [4096, 32 * 1024, 128 * 1024]
context_labels = ["4K", "32K", "128K"]

print('Read the values printed above and connect them to the concept in this cell.')
print()
print(f"{'Read the values printed above and connect them to the concept in this cell.':<8}", end="")
for prec in precisions:
    print(f"{prec:>10}", end="")
print()
print("-" * 38)

for i, seq_len in enumerate(context_lengths):
    print(f"{context_labels[i]:<8}", end="")
    for prec, bpv in precisions.items():
        size = kv_cache_bytes(seq_len, bpv)
        print(f"{format_size(size):>10}", end="")
    print()

print()
print('"Key observation："')
fp16_128k = kv_cache_bytes(128 * 1024, 2)
int4_128k = kv_cache_bytes(128 * 1024, 0.5)
print(f"Read the values printed above and connect them to the concept in this cell.{format_size(fp16_128k)}Read the values printed above and connect them to the concept in this cell.{format_size(int4_128k)}")
print(f"Read the values printed above and connect them to the concept in this cell.{(1 - int4_128k/fp16_128k)*100:.0f}Read the values printed above and connect them to the concept in this cell.")

In [5]:
# Teaching note: follow this line to see the main step.
import random

def simulate_memory(requests, block_size=16, total_memory=1024):
    'Read the values printed above and connect them to the concept in this cell.'
    
    # Teaching note: follow this line to see the main step.
    contiguous_used = 0
    contiguous_wasted = 0
    for req_len in requests:
        contiguous_used += req_len
        # Teaching note: follow this line to see the main step.
        aligned = ((req_len + 63) // 64) * 64
        contiguous_wasted += (aligned - req_len)
    
    # Teaching note: follow this line to see the main step.
    paged_used = 0
    paged_wasted = 0
    for req_len in requests:
        paged_used += req_len
        # Teaching note: follow this line to see the main step.
        paged_wasted += (block_size - (req_len % block_size)) % block_size
    
    return contiguous_wasted, paged_wasted

# Teaching note: follow this line to see the main step.
random.seed(42)
requests = [random.randint(10, 500) for _ in range(50)]

cw, pw = simulate_memory(requests)
print(f"Number of tokens: {sum(requests)}")
print(f"Read the values printed above and connect them to the concept in this cell.{cw}Read the values printed above and connect them to the concept in this cell.{cw/sum(requests)*100:.1f}%)")
print(f"Read the values printed above and connect them to the concept in this cell.{pw}Read the values printed above and connect them to the concept in this cell.{pw/sum(requests)*100:.1f}%)")
print(f"Read the values printed above and connect them to the concept in this cell.{cw-pw}Read the values printed above and connect them to the concept in this cell.")

Number of tokens: Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [6]:
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.

print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Reason')

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Reason

---

1. .2. .3. .4. .5. .6. .7. .8. .
.